# 03 Imbalance Experiment

Purpose: Simulate retrieval-time class imbalance and evaluate all scoring variants at ratios 10:1, 50:1, and 100:1.

Inputs: `dataset/CVPR_2024_dataset_Test/`, `dataset_text/test.csv`, populated `chroma_db/`.

Outputs: `results/phase2/imbalance_results.json`, `figures/phase2/minority_f1_vs_imbalance.png`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../..').resolve()))

import torch
from torchvision import models, transforms
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.auto import tqdm

from src.phase2.config import get_phase2_config
from src.phase2.data_utils import build_records_from_csv
from src.phase2.db_client import get_image_collection, get_persistent_client, get_text_collection
from src.phase2.evaluation import compute_metrics, load_image_as_numpy, save_results
from src.phase2.imbalance import simulate_imbalance
from src.phase2.scoring import global_dnds, idw, local_dnds, majority_vote, traditional, _safe_query
from src.phase2.visualization import plot_minority_f1_vs_imbalance

CONFIG = get_phase2_config()

REPO_ROOT = Path('../..').resolve()
TEST_DIR = REPO_ROOT / 'dataset' / 'CVPR_2024_dataset_Test'
TEST_CSV = REPO_ROOT / 'dataset_text' / 'test.csv'
RESULTS_PATH = REPO_ROOT / 'results' / 'phase2' / 'imbalance_results.json'
FIG_PATH = REPO_ROOT / 'figures' / 'phase2' / 'minority_f1_vs_imbalance.png'

In [ ]:
test_samples, missing_examples, total_rows = build_records_from_csv(
    csv_path=TEST_CSV,
    split_dir=TEST_DIR,
    text_column='text',
    label_column='label',
    text_key='text',
)

if missing_examples:
    print('Skipped test rows with missing image files (up to 10 shown):')
    for item in missing_examples:
        print(f'  - {item}')

if not test_samples:
    raise RuntimeError('No test samples found after image path resolution.')

print(f'Test samples available for imbalance experiment: {len(test_samples)} / {total_rows}')

client = get_persistent_client(str(REPO_ROOT / 'chroma_db'))
image_collection = get_image_collection(client)
text_collection = get_text_collection(client)
image_ckpt = REPO_ROOT / 'models' / 'mobilenetv3_best.pth'
text_ckpt = REPO_ROOT / 'text_models' / 'distilbert_best.pth'
traditional_ready = False
traditional_kwargs = {}
if image_ckpt.exists() and text_ckpt.exists():
    image_model = models.mobilenet_v3_large(weights=None)
    image_model.classifier[3] = torch.nn.Linear(image_model.classifier[3].in_features, len(CONFIG['class_names']))
    img_payload = torch.load(image_ckpt, map_location='cpu')
    image_model.load_state_dict(img_payload.get('model_state_dict', img_payload))
    image_model.eval()

    text_model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=len(CONFIG['class_names']))
    txt_payload = torch.load(text_ckpt, map_location='cpu')
    text_model.load_state_dict(txt_payload.get('model_state_dict', txt_payload))
    text_model.eval()

    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    image_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    traditional_kwargs = {
        'image_model': image_model,
        'text_model': text_model,
        'tokenizer': tokenizer,
        'transform': image_transform,
    }
    traditional_ready = True

variants = {
    'majority_vote': majority_vote,
    'idw': idw,
    'global_dnds': global_dnds,
    'local_dnds': local_dnds,
}

In [ ]:
all_results = {'ratios': CONFIG['imbalance_ratios'], 'variants': {k: {} for k in variants}}

for ratio in CONFIG['imbalance_ratios']:
    for name, fn in variants.items():
        y_true, y_pred = [], []
        for sample in tqdm(test_samples, desc=f'{name} @ {ratio}:1'):
            query_image = load_image_as_numpy(sample['image_path'])
            raw_image, raw_text = _safe_query(
                image_collection, text_collection, query_image, sample['text'], CONFIG['K_density']
            )
            filtered_image = simulate_imbalance(raw_image, CONFIG['majority_classes'], CONFIG['minority_classes'], ratio)
            filtered_text = simulate_imbalance(raw_text, CONFIG['majority_classes'], CONFIG['minority_classes'], ratio)
            pred = fn(
                query_image=query_image,
                query_text=sample['text'],
                image_collection=image_collection,
                text_collection=text_collection,
                config=CONFIG,
                alpha=CONFIG['alpha'],
                raw_image_results=filtered_image,
                raw_text_results=filtered_text,
            )
            y_true.append(sample['label'])
            y_pred.append(pred)

        metrics = compute_metrics(y_true, y_pred, CONFIG['class_names'])
        all_results['variants'][name][str(ratio)] = metrics

    if traditional_ready:
        t_true, t_pred = [], []
        for sample in tqdm(test_samples, desc=f'traditional @ {ratio}:1'):
            query_image = load_image_as_numpy(sample['image_path'])
            pred = traditional(
                query_image=query_image,
                query_text=sample['text'],
                image_collection=image_collection,
                text_collection=text_collection,
                config=CONFIG,
                alpha=CONFIG['alpha'],
                **traditional_kwargs,
            )
            t_true.append(sample['label'])
            t_pred.append(pred)
        all_results['variants'].setdefault('traditional', {})[str(ratio)] = compute_metrics(t_true, t_pred, CONFIG['class_names'])

In [ ]:
save_results(all_results, str(RESULTS_PATH))
plot_minority_f1_vs_imbalance(all_results, str(FIG_PATH))
all_results